In [ ]:
# ==========================================
# 🚀 IMPORTS
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

sns.set(style="darkgrid")

# ==========================================
# 📂 LOAD DATA (YOUR DATASET)
# ==========================================
df = pd.read_csv("/content/fear_greed_index.csv")

# ==========================================
# 🧹 PREPROCESSING
# ==========================================
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')

# Encode sentiment
sentiment_map = {
    'Extreme Fear': 0,
    'Fear': 1,
    'Neutral': 2,
    'Greed': 3,
    'Extreme Greed': 4
}
df['sentiment_score'] = df['classification'].map(sentiment_map)

# ==========================================
# ⚙️ SIMULATE TRADER DATA (REALISTIC)
# ==========================================
np.random.seed(42)

n_traders = 200

traders = []

for trader in range(n_traders):
    for _, row in df.iterrows():

        sentiment = row['sentiment_score']

        # Behavior logic (VERY IMPORTANT 🔥)
        leverage = np.random.normal(2 + sentiment, 1)
        trade_size = np.random.normal(1000 + sentiment*300, 200)

        # PnL depends on sentiment (key insight)
        pnl = np.random.normal(sentiment*50 - 50, 100)

        traders.append([
            trader,
            row['date'],
            max(leverage, 0.5),
            max(trade_size, 100),
            pnl,
            sentiment
        ])

trade_df = pd.DataFrame(traders, columns=[
    'trader_id', 'date', 'leverage', 'trade_size', 'pnl', 'sentiment_score'
])

# ==========================================
# 🏗️ FEATURE ENGINEERING
# ==========================================
trade_df['win'] = trade_df['pnl'] > 0

# Daily metrics
daily = trade_df.groupby('date').agg({
    'pnl': 'sum',
    'trade_size': 'mean',
    'leverage': 'mean',
    'win': 'mean'
}).rename(columns={'win': 'win_rate'}).reset_index()

# ==========================================
# 📊 SEGMENTATION
# ==========================================
trade_df['leverage_group'] = np.where(trade_df['leverage'] > trade_df['leverage'].median(), 'High', 'Low')

trade_counts = trade_df.groupby('trader_id').size()
trade_df['freq_group'] = trade_df['trader_id'].map(lambda x: 'Frequent' if trade_counts[x] > trade_counts.median() else 'Infrequent')

win_rate_trader = trade_df.groupby('trader_id')['win'].mean()
trade_df['consistency_group'] = trade_df['trader_id'].map(lambda x: 'Consistent' if win_rate_trader[x] > 0.55 else 'Inconsistent')

# ==========================================
# 📊 ANALYSIS (FEAR VS GREED)
# ==========================================
analysis = trade_df.groupby('sentiment_score').agg({
    'pnl': 'mean',
    'win': 'mean',
    'leverage': 'mean',
    'trade_size': 'mean'
})

print("\n📊 Sentiment Analysis:\n", analysis)

# ==========================================
# 🤖 ML MODEL (PREDICT PROFITABILITY)
# ==========================================
trade_df['profit_class'] = np.where(trade_df['pnl'] > 0, 1, 0)

X = trade_df[['sentiment_score', 'leverage', 'trade_size']]
y = trade_df['profit_class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

preds = model.predict(X_test)

print("\n🤖 Model Performance:\n")
print(classification_report(y_test, preds))

# ==========================================
# 🔍 CLUSTERING (TRADER TYPES)
# ==========================================
kmeans = KMeans(n_clusters=3)
trade_df['cluster'] = kmeans.fit_predict(trade_df[['leverage', 'trade_size', 'pnl']])

# ==========================================
# 💾 SAVE FILES
# ==========================================
trade_df.to_csv("processed_trader_data.csv", index=False)
daily.to_csv("daily_metrics.csv", index=False)


📊 Sentiment Analysis:
                         pnl       win  leverage   trade_size
sentiment_score                                             
0                -49.774169  0.308484  2.026955   998.812195
1                 -0.354438  0.499014  3.002452  1299.758172
2                 49.265623  0.687816  4.001811  1599.125997
3                100.015252  0.840964  5.002046  1899.344947
4                149.876690  0.932040  6.000854  2200.255256

🤖 Model Performance:

              precision    recall  f1-score   support

           0       0.57      0.53      0.55     39365
           1       0.73      0.76      0.75     66395

    accuracy                           0.67    105760
   macro avg       0.65      0.64      0.65    105760
weighted avg       0.67      0.67      0.67    105760



In [ ]:
from google.colab import files
files.download("processed_trader_data.csv")
files.download("daily_metrics.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>